In [ ]:
import json
import os
from abc import ABC, abstractmethod

# =====================================================================
# 1. ABSTRACTION (Soyutlama) ve TEMEL SINIF
# =====================================================================
class SistemUyesi(ABC):
    """
    Sistemdeki tüm kişilerin (Öğrenci, Öğretmen vb.) türeyeceği soyut temel sınıf.
    """
    def __init__(self, id_no, ad, soyad):
        self._id_no = id_no  # Encapsulation için protected üye
        self.ad = ad
        self.soyad = soyad

    @abstractmethod
    def bilgileri_goster(self):
        """Her alt sınıf bu metodu kendine göre ezmek (override) zorundadır."""
        pass


# =====================================================================
# 2. INHERITANCE (Kalıtım) ve ENCAPSULATION (Kapsülleme)
# =====================================================================
class Ogretmen(SistemUyesi):
    """SistemUyesi sınıfından türetilmiş Öğretmen sınıfı."""
    def __init__(self, id_no, ad, soyad, brans):
        super().__init__(id_no, ad, soyad)
        self.brans = brans

    def bilgileri_goster(self):
        # Polymorphism örneği: Temel sınıftaki soyut metodun gerçekleştirimi
        return f"Öğretmen: {self.ad} {self.soyad} - Branş: {self.brans}"


class Ogrenci(SistemUyesi):
    """SistemUyesi sınıfından türetilmiş Öğrenci sınıfı."""
    def __init__(self, id_no, ad, soyad):
        super().__init__(id_no, ad, soyad)
        # ENCAPSULATION: Notlar listesi dışarıdan doğrudan değiştirilemesin diye private yapıldı.
        self.__notlar = []

    # Getter metodu: Özel (private) veriye güvenli erişim sağlar
    def get_notlar(self):
        return self.__notlar

    # Setter metodu: Veri eklerken kontrol mekanizması sağlar
    def not_ekle(self, not_nesnesi):
        if isinstance(not_nesnesi, Not):
            self.__notlar.append(not_nesnesi)
        else:
            raise ValueError("Eklenecek nesne 'Not' sınıfından olmalıdır!")

    # POLYMORPHISM (Çok Biçimlilik) Örneği
    def bilgileri_goster(self):
        return f"Öğrenci No: {self._id_no} - Ad Soyad: {self.ad} {self.soyad}"

    def ortalama_hesapla(self):
        """Öğrencinin tüm derslerdeki notlarının genel ortalamasını hesaplar."""
        if not self.__notlar:
            return 0.0
        toplam = sum([n.get_not_degeri() for n in self.__notlar])
        return toplam / len(self.__notlar)


# =====================================================================
# 3. DİĞER GEREKLİ SINIFLAR (Ders ve Not Sınıfları)
# =====================================================================
class Ders:
    """Okulda verilen dersleri temsil eden sınıf."""
    def __init__(self, ders_kodu, ders_adi, ogretmen):
        self.ders_kodu = ders_kodu
        self.ders_adi = ders_adi
        self.ogretmen = ogretmen  # Ogretmen nesnesi alır


class Not:
    """Öğrencilerin derslere ait notlarını tutan sınıf."""
    def __init__(self, ders, not_degeri):
        self.ders = ders  # Ders nesnesi alır
        # ENCAPSULATION: Not değerinin 0-100 arasında olması kontrol edilir
        if 0 <= not_degeri <= 100:
            self.__not_degeri = not_degeri
        else:
            raise ValueError("Not değeri 0 ile 100 arasında olmalıdır!")

    def get_not_degeri(self):
        return self.__not_degeri


# =====================================================================
# 4. OKUL SİSTEMİ (Ana Yönetim ve Dosya İşlemleri Sınıfı)
# =====================================================================
class OkulSistemi:
    """Tüm otomasyon mantığını, listeleri ve dosya yönetimini içeren sınıf."""
    def __init__(self):
        # Liste ve Sözlük (Dictionary) kullanımı
        self.ogrenciler = {}  # {ogrenci_no: Ogrenci Nesnesi}
        self.dersler = {}     # {ders_kodu: Ders Nesnesi}
        self.ogretmenler = {} # {ogretmen_id: Ogretmen Nesnesi}

    def ogrenci_ekle(self, ogrenci_no, ad, soyad):
        if ogrenci_no in self.ogrenciler:
            print("[-] Bu numaraya sahip bir öğrenci zaten kayıtlı!")
            return False
        self.ogrenciler[ogrenci_no] = Ogrenci(ogrenci_no, ad, soyad)
        print("[+] Öğrenci başarıyla eklendi.")
        return True

    def ders_ekle(self, ders_kodu, ders_adi, ogretmen_id):
        if ders_kodu in self.dersler:
            print("[-] Bu kodla bir ders zaten mevcut!")
            return False
        if ogretmen_id not in self.ogretmenler:
            print("[-] Öğretmen bulunamadı! Önce öğretmeni sisteme eklemelisiniz.")
            # Test kolaylığı için otomatik öğretmen oluşturma adımı:
            print("[*] Test amaçlı geçici bir öğretmen oluşturuluyor...")
            self.ogretmenler[ogretmen_id] = Ogretmen(ogretmen_id, "Eğitmen", "Hoca", "Genel")

        ogretmen_nesnesi = self.ogretmenler[ogretmen_id]
        self.dersler[ders_kodu] = Ders(ders_kodu, ders_adi, ogretmen_nesnesi)
        print("[+] Ders başarıyla eklendi.")
        return True

    def not_gir(self, ogrenci_no, ders_kodu, not_degeri):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı!")
            return False
        if ders_kodu not in self.dersler:
            print("[-] Ders bulunamadı!")
            return False

        try:
            not_nesnesi = Not(self.dersler[ders_kodu], not_degeri)
            self.ogrenciler[ogrenci_no].not_ekle(not_nesnesi)
            print("[+] Not başarıyla girildi.")
            return True
        except ValueError as e:
            print(f"[-] Hata: {e}")
            return False

    def ogrenci_notlarini_goster(self, ogrenci_no):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı!")
            return

        ogrenci = self.ogrenciler[ogrenci_no]
        print(f"\n--- {ogrenci.ad} {ogrenci.soyad} İsimli Öğrencinin Notları ---")
        notlar = ogrenci.get_not_notlar = ogrenci.get_not_notlar() if hasattr(ogrenci, 'get_not_notlar') else ogrenci.get_notlar()

        if not notlar:
            print("Henüz girilmiş bir not bulunmamaktadır.")
            return

        for n in notlar:
            print(f"Ders: {n.ders.ders_adi} ({n.ders.ders_kodu}) -> Not: {n.get_not_degeri()}")

    def ortalama_hesapla(self, ogrenci_no):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı!")
            return None
        ogrenci = self.ogrenciler[ogrenci_no]
        ort = ogrenci.ortalama_hesapla()
        print(f"[+] {ogrenci.ad} {ogrenci.soyad} isimli öğrencinin genel ortalaması: {ort:.2f}")
        return ort

    def basarili_ogrencileri_listele(self, baraj_notu=50.0):
        print(f"\n--- Genel Ortalaması {baraj_notu} ve Üzeri Olan Başarılı Öğrenciler ---")
        bulundu = False
        for ogr_no, ogr_nesnesi in self.ogrenciler.items():
            ort = ogr_nesnesi.ortalama_hesapla()
            if ort >= baraj_notu:
                print(f"No: {ogr_no} - {ogr_nesnesi.ad} {ogr_nesnesi.soyad} -> Ortalama: {ort:.2f}")
                bulundu = True
        if not bulundu:
            print("Başarılı öğrenci kriterine uyan kimse bulunamadı.")

    def dersleri_listele(self):
        print("\n--- Sistemde Kayıtlı Dersler ---")
        if not self.dersler:
            print("Sisteme henüz ders eklenmemiş.")
            return
        for kod, ders in self.dersler.items():
            print(f"Kod: {kod} | Ders Adı: {ders.ders_adi} | Öğretmen: {ders.ogretmen.ad} {ders.ogretmen.soyad}")

    def ogrenci_ara(self, arama_metni):
        print(f"\n--- '{arama_metni}' İfadesi İçeren Öğrenci Arama Sonuçları ---")
        bulundu = False
        for ogr_no, ogr in self.ogrenciler.items():
            if arama_metni.lower() in ogr.ad.lower() or arama_metni.lower() in ogr.soyad.lower() or arama_metni == ogr_no:
                print(ogr.bilgileri_goster() + f" | Genel Ortalama: {ogr.ortalama_hesapla():.2f}")
                bulundu = True
        if not bulundu:
            print("Eşleşen öğrenci bulunamadı.")

    # DOSYA İŞLEMLERİ: JSON formatında veri kaydetme
    def verileri_kaydet(self, dosya_adi="data.json"):
        try:
            data = {
                "ogretmenler": {},
                "dersler": {},
                "ogrenciler": {}
            }

            # Öğretmenleri serileştirme
            for k, v in self.ogretmenler.items():
                data["ogretmenler"][k] = {"ad": v.ad, "soyad": v.soyad, "brans": v.brans}

            # Dersleri serileştirme
            for k, v in self.dersler.items():
                data["dersler"][k] = {"ders_adi": v.ders_adi, "ogretmen_id": v.ogretmen._id_no}

            # Öğrencileri ve private not yapısını serileştirme
            for k, v in self.ogrenciler.items():
                not_listesi = []
                for n in v.get_notlar():
                    not_listesi.append({
                        "ders_kodu": n.ders.ders_kodu,
                        "not_degeri": n.get_not_degeri()
                    })
                data["ogrenciler"][k] = {
                    "ad": v.ad,
                    "soyad": v.soyad,
                    "notlar": not_listesi
                }

            with open(dosya_adi, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            print(f"[+] Tüm veriler '{dosya_adi}' dosyasına başarıyla kaydedildi.")
        except Exception as e:
            print(f"[-] Dosya kaydedilirken beklenmeyen hata oluştu: {e}")

    # DOSYA İŞLEMLERİ: JSON formatından veri yükleme
    def verileri_yukle(self, dosya_adi="data.json"):
        if not os.path.exists(dosya_adi):
            print(f"[-] '{dosya_adi}' dosyası bulunamadı! Yüklenecek veri yok.")
            return False

        try:
            with open(dosya_adi, "r", encoding="utf-8") as f:
                data = json.load(f)

            self.ogretmenler.clear()
            self.dersler.clear()
            self.ogrenciler.clear()

            # Öğretmenleri geri yükle
            for k, v in data.get("ogretmenler", {}).items():
                self.ogretmenler[k] = Ogretmen(k, v["ad"], v["soyad"], v["brans"])

            # Dersleri geri yükle
            for k, v in data.get("dersler", {}).items():
                self.ders_ekle(k, v["ders_adi"], v["ogretmen_id"])

            # Öğrencileri ve notlarını geri yükle
            for k, v in data.get("ogrenciler", {}).items():
                self.ogrenciler[k] = Ogrenci(k, v["ad"], v["soyad"])
                for n in v.get("notlar", []):
                    self.not_gir(k, n["ders_kodu"], n["not_degeri"])

            print(f"[+] Veriler '{dosya_adi}' dosyasından başarıyla sisteme yüklendi.")
            return True
        except Exception as e:
            print(f"[-] Dosya yüklenirken hata oluştu: {e}")
            return False


# =====================================================================
# 5. MENÜ TABANLI TERMİNAL UYGULAMASI (Main Akışı)
# =====================================================================
def ana_menu():
    sistem = OkulSistemi()

    # Sunum kolaylığı için başlangıçta örnek hazır veriler ekleyelim
    sistem.ogretmenler["T1"] = Ogretmen("T1", "Merve", "Hocam", "Yapay Zeka")

    while True:
        print("\n" + "="*40)
        print("   ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ   ")
        print("="*40)
        print("1- Öğrenci Ekle")
        print("2- Ders Ekle")
        print("3- Not Gir")
        print("4- Öğrenci Notlarını Görüntüle")
        print("5- Ortalama Hesapla")
        print("6- Başarılı Öğrencileri Listele")
        print("7- Dersleri Listele")
        print("8- Öğrenci Ara")
        print("9- Verileri Kaydet")
        print("10- Verileri Yükle")
        print("0- Çıkış")
        print("="*40)

        # TRY-EXCEPT ile menü giriş hatası yönetimi
        try:
            secim = input("Lütfen yapmak istediğiniz işlemi seçin (0-10): ").strip()

            if secim == "1":
                no = input("Öğrenci Numarası: ").strip()
                ad = input("Öğrenci Adı: ").strip()
                soyad = input("Öğrenci Soyadı: ").strip()
                if no and ad and soyad:
                    sistem.ogrenci_ekle(no, ad, soyad)
                else:
                    print("[-] Alanlar boş bırakılamaz!")

            elif secim == "2":
                kod = input("Ders Kodu (Örn: YZ101): ").strip().upper()
                ad = input("Ders Adı: ").strip()
                ogretmen_id = input("Öğretmen ID (Varsayılan için T1 girin): ").strip()
                if kod and ad:
                    sistem.ders_ekle(kod, ad, ogretmen_id)
                else:
                    print("[-] Alanlar boş bırakılamaz!")

            elif secim == "3":
                no = input("Öğrenci Numarası: ").strip()
                kod = input("Ders Kodu: ").strip().upper()
                try:
                    not_deg = int(input("Not Değeri (0-100): ").strip())
                    sistem.not_gir(no, kod, not_deg)
                except ValueError:
                    print("[-] Hata: Not değeri sayısal bir tam sayı olmalıdır!")

            elif secim == "4":
                no = input("Notlarını görmek istediğiniz Öğrenci No: ").strip()
                sistem.ogrenci_notlarini_goster(no)

            elif secim == "5":
                no = input("Ortalamasını hesaplamak istediğiniz Öğrenci No: ").strip()
                sistem.ortalama_hesapla(no)

            elif secim == "6":
                try:
                    baraj = float(input("Başarı baraj notunu girin (Varsayılan 50): ") or 50)
                    sistem.basarili_ogrencileri_listele(baraj)
                except ValueError:
                    print("[-] Hata: Geçerli bir baraj notu giriniz!")

            elif secim == "7":
                sistem.dersleri_listele()

            elif secim == "8":
                kelime = input("Aramak istediğiniz öğrenci adı/soyadı/numarası: ").strip()
                if kelime:
                    sistem.ogrenci_ara(kelime)
                else:
                    print("[-] Arama kelimesi boş olamaz!")

            elif secim == "9":
                sistem.verileri_kaydet()

            elif secim == "10":
                sistem.verileri_yukle()

            elif secim == "0":
                print("[*] Sistemden çıkış yapılıyor. İyi günler!")
                break
            else:
                print("[-] Geçersiz seçim! Lütfen 0 ile 10 arasında bir değer girin.")

        except Exception as e:
            print(f"[-] Beklenmeyen bir genel hata oluştu: {e}")

# Uygulamayı başlat
if __name__ == "__main__":
    ana_menu()


   ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ   
1- Öğrenci Ekle
2- Ders Ekle
3- Not Gir
4- Öğrenci Notlarını Görüntüle
5- Ortalama Hesapla
6- Başarılı Öğrencileri Listele
7- Dersleri Listele
8- Öğrenci Ara
9- Verileri Kaydet
10- Verileri Yükle
0- Çıkış
Lütfen yapmak istediğiniz işlemi seçin (0-10): 10
[+] Ders başarıyla eklendi.
[+] Veriler 'data.json' dosyasından başarıyla sisteme yüklendi.

   ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ   
1- Öğrenci Ekle
2- Ders Ekle
3- Not Gir
4- Öğrenci Notlarını Görüntüle
5- Ortalama Hesapla
6- Başarılı Öğrencileri Listele
7- Dersleri Listele
8- Öğrenci Ara
9- Verileri Kaydet
10- Verileri Yükle
0- Çıkış
Lütfen yapmak istediğiniz işlemi seçin (0-10): 7

--- Sistemde Kayıtlı Dersler ---
Kod: YZ101 | Ders Adı: programlama | Öğretmen: Eğitmen Hoca

   ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ   
1- Öğrenci Ekle
2- Ders Ekle
3- Not Gir
4- Öğrenci Notlarını Görüntüle
5- Ortalama Hesapla
6- Başarılı Öğrencileri Listele
7- Dersleri Listele
8- Öğrenci Ara
9- Verileri Kaydet
10